# Quick start: attributions and interactions across a Rashomon set

Train a reference model, sample its mask-based Rashomon set, and look at how feature main effects and feature interaction scores (FIS) vary across the set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

from generalized_rashomon_set import RashomonExplainer, plots

## A synthetic regression with one true interaction

`y = x0 + x1 + 2 x0 x1 + 0.5 x2 + noise`; `x3` is pure noise.

In [ ]:
rng = np.random.default_rng(0)
X = pd.DataFrame(rng.normal(size=(1500, 4)), columns=['x0', 'x1', 'x2', 'x3'])
y = X.x0 + X.x1 + 2 * X.x0 * X.x1 + 0.5 * X.x2 + rng.normal(scale=0.1, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
model = MLPRegressor((64, 64), max_iter=2000, random_state=0).fit(X_train, y_train)

## Explain

`fit()` runs the four stages: reference attributions, greedy mask search, attributions of every sampled model, and the FIS of every joint-masked pair model. `n_jobs=-1` uses all cores.

In [ ]:
explainer = RashomonExplainer(model, X_test, y_test, loss='mse', epsilon_rate=0.1, delta=0.1,
                              shuffle_times=30, n_jobs=-1, random_state=0, verbose=True)
explainer.fit()

In [ ]:
explainer.attribution_summary().round(4)

In [ ]:
explainer.interaction_summary().round(4)

## Plots

In [ ]:
plots.swarm_plot_attribution(explainer, boxplot=True);

In [ ]:
plots.swarm_plot_interaction(explainer, boxplot=True);

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plots.halo_plot(explainer, ('x0', 'x1'), ax=axes[0])   # strong interaction: empirical halo far from the circle
plots.halo_plot(explainer, ('x0', 'x2'), ax=axes[1])   # no interaction: halos coincide
plt.tight_layout()

In [ ]:
plots.search_trace_plot(explainer);

## Several Rashomon radii in one plot

Pass `radii` to `explain_interactions` to evaluate pair models whose single-feature loss budgets add up to a fraction of `epsilon`.

In [ ]:
explainer.explain_interactions(order=2, radii=(0.2, 0.4, 0.6, 0.8, 1.0))
plots.halo_plot(explainer, ('x0', 'x1'));

## Save and reload

In [ ]:
explainer.save('quickstart_results.json')
reloaded = RashomonExplainer.load('quickstart_results.json', model, X_test, y_test)
reloaded.interaction_summary().head()